# 模組 6 · 模型部署
## 儲存/載入/預測、匯出 .n4a、工作區管理、sklearn 整合

把訓練好的模型變成可重複使用、可分享、可上線的成品。

**對應官方範例**：`examples/user/06_deployment/`（U01–U04）

## 🚀 環境設定（請先執行下面這格）

本課程使用 **nirs4all 官方範例資料集（sample_data）**。下面這格會：
1. 從 GitHub 下載 nirs4all（內含 0.9 版函式庫與官方光譜資料集）
2. 安裝函式庫
3. 切換到 `examples/` 目錄（裡面有 `sample_data/`）

> 💡 **為什麼從 GitHub 安裝？** PyPI（`pip install nirs4all`）目前是舊版 0.4，沒有本課程使用的 `nirs4all.run()` 簡易 API。GitHub 上的版本（0.9+）才有，且需要 **Python ≥ 3.11**（Colab 預設 3.12，沒問題）。

第一次執行約需 1–2 分鐘，請耐心等候出現「✅ 完成」。

In [ ]:
# === Colab / Jupyter 環境設定（每本 notebook 開頭執行一次）===
import os, sys, subprocess

if not os.path.isdir('nirs4all'):
    print('⬇️  下載 nirs4all 與官方資料集（約 1–2 分鐘）…')
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/GBeurier/nirs4all.git'], check=True)
    print('📦 安裝 nirs4all …')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    './nirs4all'], check=True)

# 切換到 examples 目錄（內含 sample_data）
if os.path.basename(os.getcwd()) != 'examples' and os.path.isdir('nirs4all/examples'):
    os.chdir('nirs4all/examples')

import nirs4all
print('✅ 完成！nirs4all 版本：', nirs4all.__version__)
print('   工作目錄：', os.getcwd())
print('   可用資料集：', [d for d in os.listdir('sample_data') if os.path.isdir(os.path.join('sample_data', d))])

---
## U01 · 儲存、載入與預測  ★★☆☆☆

用 `save_artifacts=True` 訓練後，模型（含前處理鏈）會存到 workspace。本課示範訓練後直接用 `result` 取得最佳模型資訊。新資料預測可用預測項目或 model ID。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import StandardNormalVariate, FirstDerivative

pipeline = [
    MinMaxScaler(), StandardNormalVariate(), FirstDerivative(),
    {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=5),  "name": "PLS-5"},
    {"model": PLSRegression(n_components=10), "name": "PLS-10"},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="SaveLoad", verbose=1, save_artifacts=True)

best = result.best
print("\n最佳模型：", best.get('model_name'))
print("最佳 RMSE：", round(result.best_rmse, 4))
print("💡 模型已存到 workspace/，可用 nirs4all.predict() 對新資料預測。")

> ✍️ **練習**：用 `result.top(1)[0]` 取得最佳預測項目，研究它包含哪些欄位（model_name、id、preprocessings…）。

---
## U02 · 匯出 Bundle：可攜式部署  ★★☆☆☆

把模型打包成可攜檔。`.n4a`：支援所有模型型態但需 nirs4all；`.n4a.py`：僅 sklearn 的獨立腳本（只需 numpy + joblib）。用 `result.export(path)` 匯出。

In [ ]:
import nirs4all, os
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import RepeatedKFold
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import StandardNormalVariate, SavitzkyGolay

pipeline = [
    MinMaxScaler(), {"y_processing": MinMaxScaler()},
    {"feature_augmentation": [StandardNormalVariate(), SavitzkyGolay()]},
    RepeatedKFold(n_splits=2, n_repeats=1, random_state=42),
    {"model": PLSRegression(n_components=10), "name": "PLS-10"},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="Export", verbose=1, save_artifacts=True)

# 匯出最佳模型為 .n4a bundle
os.makedirs("exports", exist_ok=True)
bundle_path = result.export("exports/best_model.n4a")
print("\n✅ 已匯出：", bundle_path)
print("檔案大小：", round(os.path.getsize(bundle_path)/1024, 1), "KB")

> ✍️ **練習**：把 `format` 改成匯出 `.n4a.py` 可攜腳本，檢視它是否只依賴 numpy + joblib。

---
## U03 · 工作區管理：Session  ★★☆☆☆

用 `nirs4all.session()` context manager 共享設定並組織多次 run，最後收集所有結果挑最佳。工作區結構：`store.sqlite`（結構化資料）+ `artifacts/` + `exports/` + `library/`。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge

cv = ShuffleSplit(n_splits=3, test_size=0.25, random_state=42)
results = []
with nirs4all.session(verbose=1, save_artifacts=True) as s:
    results.append(nirs4all.run(
        pipeline=[MinMaxScaler(), {"y_processing": MinMaxScaler()}, cv,
                  {"model": PLSRegression(n_components=10)}],
        dataset="sample_data/regression", name="PLS", session=s))
    results.append(nirs4all.run(
        pipeline=[MinMaxScaler(), {"y_processing": MinMaxScaler()}, cv,
                  {"model": Ridge(alpha=1.0)}],
        dataset="sample_data/regression", name="Ridge", session=s))

best = min(results, key=lambda r: r.best_rmse)
print("\n全域最佳 RMSE:", round(best.best_rmse, 4))

> ✍️ **練習**：在 session 內多跑一個資料集，最後用 `min(...)` 找出跨設定的全域最佳。

---
## U04 · sklearn 整合：NIRSPipeline  ★★★☆☆

`NIRSPipeline` 把訓練好的模型包裝成 sklearn 相容物件，可用 `predict()`、`score()`、`transform()`，並接上 `cross_val_score`、`GridSearchCV`、SHAP。

In [ ]:
import nirs4all
from nirs4all.sklearn import NIRSPipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import StandardNormalVariate

result = nirs4all.run(
    pipeline=[StandardNormalVariate(), MinMaxScaler(), {"y_processing": MinMaxScaler()},
              ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
              {"model": PLSRegression(n_components=10)}],
    dataset="sample_data/regression", name="SklearnWrap", verbose=1)

# 包裝成 sklearn 相容物件
pipe = NIRSPipeline.from_result(result)
print("\nNIRSPipeline 提供：predict() / score() / transform()")
print("底層模型：", type(pipe.model_).__name__ if hasattr(pipe, 'model_') else "—")

> ✍️ **練習**：把 `pipe` 餵進 sklearn 的 `cross_val_score`，得到跨折 R² 分佈。